# Workshop 5. A practice case on a customer survey

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture05/workshop05.ipynb)

This workshop is a practice run for the exam assignment. The
notebook has the same structure as sections 1 to 4 of the assignment: the case, data preparation, exploratory
analysis, and estimation and tests. The questions are numbered like the assignment's, and each has a code cell and an
answer cell. Work in your group if you have one; this is a good occasion to try working together on a notebook.

The data are a second RetailCorp file with the same kinds of problems as the lecture file, on other variables. The
lecture notebook shows every step in code, and the code patterns below cover what is new. Under each question there
are two hints: the first names the step, the second names the functions. Try the question before you open them.
You are welcome to use an LLM, but every question can be solved without one, and you should be able to explain each
decision yourself: at the oral exam you will be asked about the choices your group made.

Start with the questions that are not marked; they cover the whole path from raw file to test. Questions marked
*(if time)* are good practice for the assignment, and you can return to them after the workshop.

**Updated 10 November 2026.** The note on the cleaning log below has been corrected. Its last example was missing
`.sum()`, so it did not give a count, and its examples used columns that are not in this data. The examples now use
the columns of this workshop, and the note adds two points: count the rows before you fix a problem, and empty cells
are counted by `~isin`. The solutions notebook shows how each count in the cleaning log is computed.

## Toolbox

| Function | What it does |
|---|---|
| `df.info()`, `df.describe()`, `df["col"].value_counts(dropna=False)` | Types and non-missing counts; numerical summary; the values in a column, including `NaN` |
| `df.duplicated().sum()`, `df.drop_duplicates()` | Count and remove rows that are exact copies of an earlier row |
| `df["col"].str.strip().str.lower()` | Remove spaces at both ends and use lower case |
| `df["col"].str.replace(old, new, regex=False)` | Replace plain text |
| `df["col"].map(dictionary)` | Translate each value; values not in the dictionary become `NaN` |
| `.astype(float)`, `.astype(int)` | Convert; stops with an error if a value does not fit. `int` cannot hold `NaN` |
| `df["col"].replace(code, np.nan)` | Turn a code for missing into `NaN` |
| `df["col"].mask(condition)` | Set the values where the condition is true to `NaN` |
| `df.isna().sum()`, `df["col"].isna().groupby(df["group"]).mean()` | Missing values per column; share missing in each group |
| `df.groupby("group")["col"].agg(["count", "mean", "median"])` | Summary by group |
| `df[cols].corr()`, `plt.hist(x, bins=20)`, `stats.skew(x)` | Correlations, histogram, skewness |
| `stats.t.ppf`, `stats.t.sf`, `stats.norm.ppf` | Critical values and p-values, as in Lectures 2 to 4 |

## Code patterns

Three patterns for what is new today. They are templates and do not run as they stand.

**A number written the Danish way, to float**

```python
text = df["col"].str.replace(" kr.", "", regex=False)    # 1. remove the unit
text = text.str.replace(".", "", regex=False)             # 2. remove the thousand separator
text = text.str.replace(",", ".", regex=False)            # 3. decimal comma to point
df["col"] = text.astype(float)                            # stops with an error if a value does not fit
print(df["col"].dtype, df["col"].isna().sum())            # check
```

**Labels to one spelling, and yes/no to 1 and 0**

```python
key = df["col"].str.strip().str.lower()
print(key.value_counts())                                 # which spellings are left?
df["col"] = key.map({"yes": 1, "no": 0}).astype(int)      # every spelling must be in the dictionary
```

**The cleaning log**

```python
log = []
log.append(("Problem", rows_affected, "Decision", "Consequence if not handled"))
pd.DataFrame(log, columns=["Problem", "Rows", "Decision", "Consequence if not handled"])
```

The tests in section 4 follow the code in the lecture notebooks for Lectures 2 to 4.

<div style="border: 1px solid #9ca3af; border-radius: 8px; padding: 1rem 1.25rem; margin: 0.5rem 0;">

## A quick note on the cleaning log

At Workshop 5, you were asked to keep a log of the changes made during data cleaning. The log helps you explain what you found, what you decided to do, and why the decision matters.

A useful structure is:

| Problem | Rows affected | Decision | Consequence if not handled |
|---|---:|---|---|
| What was wrong with the data? | How many rows had the problem? | What did you do about it? | What could go wrong if you left it as-is? |

The problem, decision, and consequence depend on the data and your reasoning. For **Rows affected**, count the rows that have the problem. Finding that count is also a useful way to check that you have identified the relevant rows correctly.

Count the rows **before** you fix a problem. After `drop_duplicates()`, or after the labels have been translated, the same count gives 0.

### Example: counting duplicates

For exact duplicate rows, count them before removing them:

```python
n_dup = df.duplicated().sum()
```

`duplicated()` marks repeated rows as `True` (excluding the first occurrence), and `sum()` counts those `True` values. You can then record `n_dup` in the log alongside your description, decision, and consequence. Alternatively, compare the number of rows before and after removing duplicates.

### Counting rows with conditions

Pandas conditions produce a Boolean Series (`True`/`False`). Because `True` counts as 1 and `False` as 0, summing the Series counts matching rows. The examples use the `region` and `channel` columns of this workshop's data.

Use `.isin([...])` to check whether a column's value is one of several values:

```python
n_north = df["region"].isin(["Nordjylland", "Midtjylland"]).sum()
```

This counts rows where `region` is either Nordjylland or Midtjylland, spelled exactly like that.

Use `~` (tilde) to negate a condition. Put the condition in parentheses so the negation happens before `.sum()`:

```python
regions = ["Hovedstaden", "Midtjylland", "Syddanmark", "Sjælland", "Nordjylland"]
n_other = (~df["region"].isin(regions)).sum()
```

This counts rows where `region` is none of the five correct spellings: the rows with a label problem.

Use `|` for **OR**. It combines conditions row by row, so each matching row is counted once, even if it matches both:

```python
n_rows = (
    ~df["region"].isin(regions) |
    ~df["channel"].isin(["App", "Web"])
).sum()
```

This counts rows with a label problem in `region` **or** in `channel`.

Be careful when negating multiple conditions. Take the rows where the channel is App **or** the region is Nordjylland:

```python
df["channel"].isin(["App"]) | df["region"].isin(["Nordjylland"])
```

This expression:

```python
~df["channel"].isin(["App"]) | ~df["region"].isin(["Nordjylland"])
```

means the channel is not App **or** the region is not Nordjylland. It is **not** the opposite of the earlier OR condition. To count rows that satisfy neither condition, negate the combined condition and then sum:

```python
n_neither = (~(
    df["channel"].isin(["App"]) |
    df["region"].isin(["Nordjylland"])
)).sum()
```

This is equivalent to requiring both that the channel is not App **and** that the region is not Nordjylland.

Watch out for empty cells: `isin` gives `False` for an empty cell, so `~df["col"].isin([...])` also counts the rows where the value is missing. If a column has missing values, count them separately with `df["col"].isna().sum()`, and count the wrong labels with `(~df["col"].isin([...]) & df["col"].notna()).sum()`.

With these patterns, you should be able to identify and count the rows affected by each cleaning decision.

</div>

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
raw = pd.read_csv(url + "customer_survey.csv")
alpha = 0.05
print(f"Rows: {len(raw)}, columns: {raw.shape[1]}")

## 1 The case

In September 2026 RetailCorp invited 1,500 customers to a short survey. They were drawn at random from all customers
who ordered at least once between January and June 2026. 600 answered. IT linked each answer to the customer's orders
and spend in January to June. Management asks three questions:

1. What share of customers would recommend RetailCorp to a friend?
2. Is average satisfaction above the target of 7 on the 0 to 10 scale?
3. Do members of the loyalty club spend more than other customers?

The data description from IT:

| Column | Content |
|---|---|
| `customer_id` | Customer number |
| `age` | Age in years |
| `region` | The customer's region |
| `channel` | Where the customer orders most often: App or Web |
| `member` | Member of the loyalty club |
| `orders_h1` | Number of orders, January to June 2026 |
| `spend_h1` | Total spend in DKK, January to June 2026, as exported from the shop system |
| `satisfaction` | "How satisfied are you with RetailCorp?", 0 to 10. "Don't know" is recorded as −1; empty if the question was skipped |
| `recommend` | "Would you recommend RetailCorp to a friend?" |

**1.1** Restate the three questions in your own words. Which population can the answers describe, and who is not in the
data? What could that mean for the results?

<details>
<summary>Hint 1</summary>

Think of the steps from all customers to the 600 rows: who was invited, and who answered. Newbold 1.1 describes
nonresponse.
</details>

<details>
<summary>Hint 2</summary>

No code is needed. Compare the 1,500 invited with the 600 who answered.
</details>

**Your answer:**

## 2 Data preparation

Keep the raw data in `raw` and clean a copy, `df`. Follow the sequence from the lecture for every problem: look,
decide, do, check, log.

**2.1** Inspect the data. The cell is given: run it and describe every problem you find. For each problem, say whether
Python would stop with an error or give a wrong number without warning.

In [ ]:
raw.info()
print(raw.describe(), "\n")
for col in ["region", "channel", "member", "recommend"]:
    print(raw[col].value_counts(), "\n")
print("Rows that are exact copies of an earlier row:", raw.duplicated().sum())
print(raw["spend_h1"].head(4).tolist())

**Your answer:**

In [ ]:
df = raw.copy()
log = []

**2.2a** Duplicate rows. Remove them and check that every customer occurs once.

<details>
<summary>Hint 1</summary>

Exact copies of a row are the same answer submitted twice; keep one copy. Check that `customer_id` is unique
afterward.
</details>

<details>
<summary>Hint 2</summary>

`df.duplicated().sum()`, `df.drop_duplicates()`, `df["customer_id"].duplicated().sum()`, `len(df)`.
</details>

**Your answer:**

**2.2b** Inconsistent labels. Give `region` and `channel` one spelling per group, and check that no label was missed.

<details>
<summary>Hint 1</summary>

Remove spaces and use lower case first, so that only the real differences remain. Then translate each spelling. A
label you forgot becomes `NaN`, so count missing values afterward.
</details>

<details>
<summary>Hint 2</summary>

`.str.strip()`, `.str.lower()`, `.str.replace("region ", "", regex=False)`, `.map({...})`,
`.value_counts(dropna=False)`. Count the rows affected with `~df["region"].isin([...])`.
</details>

**Your answer:**

**2.2c** Numbers and yes/no stored as text. Convert `spend_h1` to `float`, and `member` and `recommend` to `int` with 1 for
yes and 0 for no. Check that nothing became missing.

<details>
<summary>Hint 1</summary>

Spend: remove the unit, then the thousand separator, then turn the decimal comma into a point (in this order). Yes/no:
first one spelling, then 1 and 0.
</details>

<details>
<summary>Hint 2</summary>

The first two code patterns: `.str.replace(..., regex=False)`, `.astype(float)`, `.str.strip().str.lower()`,
`.map({...})`, `.astype(int)`, `.isna().sum()`.
</details>

**Your answer:**

**2.2d** Codes and impossible values. Handle the code −1 in `satisfaction` and the impossible ages. What happens to the mean
satisfaction and the mean age?

<details>
<summary>Hint 1</summary>

−1 is the code for "don't know" (see the data description), so it is a missing answer. An age of 0 is impossible, and
four-digit ages are birth years typed in the wrong field. Decide what to do with each and say why.
</details>

<details>
<summary>Hint 2</summary>

`.value_counts(dropna=False).sort_index()`, `.replace(-1, np.nan)`, `.describe()`,
`df.loc[condition, ["customer_id", "age"]]`, `.mask(condition)`.
</details>

**Your answer:**

**2.2e** Error or real? Look at the largest values of `spend_h1`. Which values are errors, and how can you tell them from
customers who really spend a lot? What happens to the mean and the median?

<details>
<summary>Hint 1</summary>

A large total is plausible for a customer with many orders. Put spend on the same footing for every customer by
dividing by the number of orders, and look for values far from all the others.
</details>

<details>
<summary>Hint 2</summary>

`df["spend_h1"] / df["orders_h1"]`, `.sort_values(ascending=False).head()`, `df.assign(...)`, `.mask(condition)`,
`.mean()`, `.median()`.
</details>

**Your answer:**

**2.3** Show the cleaning log as a table. Add one sentence below it: which decisions do you expect to matter most for the
three management questions, and why?

<details>
<summary>Hint 1</summary>

One row per problem: the problem, the number of rows affected, the decision, and the consequence if not handled.
</details>

<details>
<summary>Hint 2</summary>

`pd.DataFrame(log, columns=[...])` (third code pattern). `pd.set_option("display.max_colwidth", 120)` shows long text.
</details>

**Your answer:**

**2.4** Missing values. How many are there in each column after cleaning? Is satisfaction missing more often for customers
with one order than for customers with more? Which of the three types of missing data could this be, and how do you
handle the missing values in section 4?

<details>
<summary>Hint 1</summary>

Compare the share missing in two groups. Then compare the satisfaction of those who answered in the two groups: what
does that mean for the mean of all who answered?
</details>

<details>
<summary>Hint 2</summary>

`df.isna().sum()`, `one = df["orders_h1"] == 1`, `df["satisfaction"].isna().groupby(one).mean()`,
`df.groupby(one)["satisfaction"].agg(["mean", "count", "size"])`.
</details>

**Your answer:**

**2.5** *(if time)* Create the variables you need later: `spend_per_order` (spend divided by the number of orders) and `one_order` (1 if
the customer ordered once, 0 otherwise). Check that they look as you expect.

<details>
<summary>Hint 1</summary>

A new column is computed from existing columns, row by row. A true/false condition becomes 1 and 0 with `astype(int)`.
</details>

<details>
<summary>Hint 2</summary>

`df["spend_h1"] / df["orders_h1"]`, `(df["orders_h1"] == 1).astype(int)`, `.describe()`.
</details>

**Your answer:**

**2.6** *(if time)* Save the cleaned data so that you can continue in a later session, either in your Google Drive or by downloading it.
Read the saved file back and check that it matches `df`. Why must the notebook you hand in at the exam still start
from the raw file?

<details>
<summary>Hint 1</summary>

Write the file with `index=False`, read it back, and compare. Look at section "Keeping your work in Colab" in the
lecture notebook.
</details>

<details>
<summary>Hint 2</summary>

`drive.mount`, `os.makedirs`, `df.to_csv(..., index=False)`, `pd.read_csv`, `pd.testing.assert_frame_equal`; or
`files.download`. `df` still has the row numbers from before the duplicates were removed, so compare with
`df.reset_index(drop=True)`.
</details>

**Your answer:**

## 3 Exploratory analysis

Get to know the cleaned data before you test. Aim the analysis at the three questions.

**3.1** Make a table of descriptive statistics for the numerical variables. Compare the mean and the median of `spend_h1`.
What does the comparison say about the shape of the distribution?

<details>
<summary>Hint 1</summary>

When the mean is clearly above the median, a few large values pull the mean up: the distribution is skewed to the
right (Newbold 2.1).
</details>

<details>
<summary>Hint 2</summary>

`df[cols].describe()` with `cols = ["age", "orders_h1", "spend_h1", "spend_per_order", "satisfaction"]`.
</details>

**Your answer:**

**3.2** Plot the distribution of `spend_h1` and compute its skewness. Describe the shape in words.

<details>
<summary>Hint 1</summary>

A histogram with about 20 bins, with lines for the mean and the median.
</details>

<details>
<summary>Hint 2</summary>

`plt.hist(x, bins=20)`, `plt.axvline(...)`, `stats.skew(x)`; drop missing values first with `.dropna()`.
</details>

**Your answer:**

**3.3** *(if time)* Compare the groups that matter for the questions: spend by membership, and satisfaction and the share recommending by
membership and by channel. What patterns do you see?

<details>
<summary>Hint 1</summary>

For each group, the number of customers, the mean and, for spend, the median. Remember that a difference in a sample
still has to be tested.
</details>

<details>
<summary>Hint 2</summary>

`df.groupby("member")["spend_h1"].agg(["count", "mean", "median"])`,
`df.groupby("channel")[["satisfaction", "recommend"]].mean()`.
</details>

**Your answer:**

**3.4** *(if time)* Compute the correlation matrix of `age`, `orders_h1`, `spend_h1`, `satisfaction`, `recommend` and `member`. Which
correlations stand out?

<details>
<summary>Hint 1</summary>

A correlation measures a straight-line relationship between two variables, from −1 to 1 (Newbold 2.4). Look for the
largest values outside the diagonal.
</details>

<details>
<summary>Hint 2</summary>

`df[cols].corr().round(2)`.
</details>

**Your answer:**

## 4 Estimation and tests

For each test, write H<sub>0</sub> and H<sub>1</sub>, say whether the test is one-sided or two-sided and why, check
the conditions, compute the result, and give the answer in words for management. Use α = 0.05. Use the cleaned data
and the customers with a value, and report n.

**4.1** Estimate the share of customers who would recommend RetailCorp, with a 95% confidence interval. Check the condition.

<details>
<summary>Hint 1</summary>

The confidence interval for a proportion from Lecture 2: p̂ ± z<sub>α/2</sub> √(p̂(1 − p̂)/n), with the condition
np̂(1 − p̂) > 5.
</details>

<details>
<summary>Hint 2</summary>

`df["recommend"].mean()`, `.count()`, `np.sqrt`, `stats.norm.ppf(1 - alpha / 2)`.
</details>

**Your answer:**

**4.2** Is average satisfaction above the target of 7?

<details>
<summary>Hint 1</summary>

The one-sample t test from Lecture 3, one-sided: H<sub>0</sub>: μ ≤ 7 and H<sub>1</sub>: μ > 7. Use only the
customers with a satisfaction score.
</details>

<details>
<summary>Hint 2</summary>

`.dropna()`, `len`, `.mean()`, `.std(ddof=1)`, `np.sqrt`, `stats.t.ppf(1 - alpha, df=n - 1)`, `stats.t.sf`.
</details>

**Your answer:**

**4.3** Do loyalty club members spend more on average than other customers?

<details>
<summary>Hint 1</summary>

Two independent groups: the t test for two means from Lecture 4, one-sided, H<sub>1</sub>: μ<sub>member</sub> −
μ<sub>other</sub> > 0. Use Welch's version, since the spreads differ.
</details>

<details>
<summary>Hint 2</summary>

`df.loc[df["member"] == 1, "spend_h1"].dropna()`, `.var(ddof=1)`, `np.sqrt`, the Welch degrees of freedom, `stats.t.sf`.
Check with `stats.ttest_ind(x1, x2, equal_var=False, alternative="greater")`.
</details>

**Your answer:**

**4.4** *(if time)* What did the cleaning change? Run 4.2 again with the −1 codes kept as scores, and 4.3 again with the three extra
zeros kept. Which cleaning decisions changed a conclusion?

<details>
<summary>Hint 1</summary>

Start again from `raw`: remove the duplicates and redo only the steps you need, leaving out the one you want to look
at. Keep the same index as `df`, so that the member column still matches.
</details>

<details>
<summary>Hint 2</summary>

`raw.drop_duplicates()`, `.dropna()`, the conversion from 2.2c without `.mask(...)`. `df["member"]` has the same index
as `raw.drop_duplicates()`, so `spend_raw[df["member"] == 1]` selects the members.
</details>

**Your answer:**

## Extension tasks

For those who have time left. They go beyond the questions of the assignment, but they are good preparation for the
analytical discussion at the oral exam.

### Extension 1. Fill instead of drop

Fill the missing satisfaction scores with the mean and run the test in 4.2 again. What happens to n, the mean, the
standard deviation and t? Why should you not report this version?

<details>
<summary>Hint 1</summary>

Every filled value sits exactly at the mean.
</details>

<details>
<summary>Hint 2</summary>

`df["satisfaction"].fillna(df["satisfaction"].mean())`.
</details>

**Your answer:**

### Extension 2. More orders or larger orders?

Do members spend more per order than other customers? Test it with `spend_per_order` and compare with 4.3.

<details>
<summary>Hint 1</summary>

The same Welch test as in 4.3, on another variable.
</details>

<details>
<summary>Hint 2</summary>

`stats.ttest_ind(..., equal_var=False, alternative="greater")`.
</details>

**Your answer:**